# Full Workflow - Single `.mo` File

This notebook runs the complete single-file workflow: build an auxiliary model, simulate it, extract initialization values, and write the initialized dynamic model.


In [ ]:
# Cell 1: Environment Setup

include("scripts/buildaux_helpers.jl")
include("scripts/buildaux_dictionaries.jl")
include("scripts/initialization_helpers.jl")
include("scripts/initialization_dictionaries.jl")

using OMJulia
using Plots, DataFrames, CSV


In [ ]:
# Cell 2: User Configuration

# 1. Directory containing the single-file model
MODEL_DIR = abspath("models")

# 2. Select the original dynamic model to initialize
MODEL = "BESSload"

# 3. Path to the selected model file
MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

# 4. Path to the Dynawo package.mo
DYNAWO_PKG_PATH = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 5. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

# 6. INIT model selection for components with multiple INIT profiles
INIT_MODEL_BY_COMPONENT = Dict{String, String}(
    # "generatorSynchronous" => "GeneratorSynchronousInt_INIT",
)

# 7. Leave empty to disable slack-specific handling
SLACK_COMPONENT = ""

# 8. Variable to plot after the initialized simulation
PLOT_VARIABLE = "BESS.terminal.V.im"


In [ ]:
# Cell 3: Derived Output Names

AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE_PATH = joinpath(MODEL_DIR, AUX_MODEL * ".mo")

INITIALIZED_MODEL = MODEL * "_initialized"
INITIALIZED_FILE_PATH = joinpath(MODEL_DIR, INITIALIZED_MODEL * ".mo")

println("Original model:    ", MODEL_FILE_PATH)
println("Auxiliary output:  ", AUX_FILE_PATH)
println("Initialized output:", INITIALIZED_FILE_PATH)


## Build Auxiliary Model


In [ ]:
# Cell 4: Load and Check the Original Model for BuildAux

BuildAuxOMC = OMJulia.OMCSession()
BuildAuxHelpers.om_send(BuildAuxOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
BuildAuxHelpers.om_send(BuildAuxOMC, "loadModel(Complex)")
BuildAuxHelpers.om_send(BuildAuxOMC, "loadModel(ModelicaServices)")
BuildAuxHelpers.om_send(BuildAuxOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
BuildAuxHelpers.om_send(BuildAuxOMC, "loadFile(\"$MODEL_FILE_PATH\")")

BuildAuxHelpers.om_send(BuildAuxOMC, "clearMessages()")
chk_source_for_aux = BuildAuxHelpers.om_send(BuildAuxOMC, "checkModel($MODEL)", parsed = false)
println(chk_source_for_aux)


In [ ]:
# Cell 5: Build and Save the Auxiliary Model

BuildAuxHelpers.om_send(BuildAuxOMC, "deleteClass($AUX_MODEL)")
BuildAuxHelpers.om_send(BuildAuxOMC, "clearMessages()")
BuildAuxHelpers.om_send(BuildAuxOMC, "copyClass($MODEL, \"$AUX_MODEL\")")

source_components_for_aux = BuildAuxHelpers.get_all_components(BuildAuxOMC, MODEL)

BuildAuxHelpers.apply_replacements!(
    BuildAuxOMC,
    MODEL,
    AUX_MODEL,
    BuildAuxDictionaries.REPLACEMENTS,
    source_components_for_aux,
    SLACK_COMPONENT,
)

BuildAuxHelpers.delete_connections!(BuildAuxOMC, AUX_MODEL, source_components_for_aux)
BuildAuxHelpers.delete_components!(BuildAuxOMC, AUX_MODEL, source_components_for_aux)

BuildAuxHelpers.add_init_models!(
    BuildAuxOMC,
    MODEL,
    AUX_MODEL,
    BuildAuxDictionaries.INIT_MODELS,
    INIT_MODEL_BY_COMPONENT,
    source_components_for_aux,
    SLACK_COMPONENT,
)

BuildAuxHelpers.apply_LF_modifiers!(
    BuildAuxOMC,
    MODEL,
    AUX_MODEL,
    BuildAuxDictionaries.INIT_MODELS,
    source_components_for_aux,
)

BuildAuxHelpers.add_init_equations!(
    BuildAuxOMC,
    MODEL,
    AUX_MODEL,
    source_components_for_aux,
    BuildAuxDictionaries.INIT_MODELS,
    INIT_MODEL_BY_COMPONENT,
    SLACK_COMPONENT,
)

BuildAuxHelpers.om_send(BuildAuxOMC, "saveModel(\"$AUX_FILE_PATH\", $AUX_MODEL)")
BuildAuxHelpers.patch_aux_equations!(AUX_FILE_PATH, SLACK_COMPONENT; components = source_components_for_aux)

BuildAuxHelpers.om_send(BuildAuxOMC, "deleteClass($AUX_MODEL)")
BuildAuxHelpers.om_send(BuildAuxOMC, "loadFile(\"$AUX_FILE_PATH\")")
BuildAuxHelpers.om_send(BuildAuxOMC, "clearMessages()")
chk_aux = BuildAuxHelpers.om_send(BuildAuxOMC, "checkModel($AUX_MODEL)", parsed = false)
println(chk_aux)
println("Wrote auxiliary model: ", AUX_FILE_PATH)


## Extract Initialization Values


In [ ]:
# Cell 6: Load Dynamic and Auxiliary Models for Initialization

DynamicOMC = OMJulia.OMCSession()
InitializationHelpers.om_send(DynamicOMC, "loadModel(Complex)")
InitializationHelpers.om_send(DynamicOMC, "loadModel(ModelicaServices)")
InitializationHelpers.om_send(DynamicOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
InitializationHelpers.om_send(DynamicOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
InitializationHelpers.om_send(DynamicOMC, "loadFile(\"$MODEL_FILE_PATH\")")
InitializationHelpers.om_send(DynamicOMC, "clearMessages()")
println("Checking the dynamic model...")
chk_dynamic = InitializationHelpers.om_send(DynamicOMC, "checkModel($MODEL)", parsed = false)
println(chk_dynamic)

AuxiliaryOMC = OMJulia.OMCSession()
InitializationHelpers.om_send(AuxiliaryOMC, "loadModel(Complex)")
InitializationHelpers.om_send(AuxiliaryOMC, "loadModel(ModelicaServices)")
InitializationHelpers.om_send(AuxiliaryOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
InitializationHelpers.om_send(AuxiliaryOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
InitializationHelpers.om_send(AuxiliaryOMC, "loadFile(\"$AUX_FILE_PATH\")")
InitializationHelpers.om_send(AuxiliaryOMC, "clearMessages()")
println("Checking the auxiliary model...")
chk_aux_for_init = InitializationHelpers.om_send(AuxiliaryOMC, "checkModel($AUX_MODEL)", parsed = false)
println(chk_aux_for_init)


In [ ]:
# Cell 7: Simulate Auxiliary Model and Extract INIT Outputs

ModelicaSystem(AuxiliaryOMC, AUX_FILE_PATH, AUX_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
simulate(AuxiliaryOMC, resultfile = AUX_MODEL * "_res.mat")

source_components_for_init = InitializationHelpers.get_all_components(DynamicOMC, MODEL)
initializable_components = InitializationHelpers.get_initializable_components(
    source_components_for_init,
    InitializationDictionaries.INIT_PARAMS,
    INIT_MODEL_BY_COMPONENT,
)

init_values_by_component = InitializationHelpers.extract_all_initialization_values(
    AuxiliaryOMC,
    source_components_for_init,
    InitializationDictionaries.INIT_PARAMS,
    INIT_MODEL_BY_COMPONENT,
)

println("Extracted initialization values for ", length(init_values_by_component), " component(s).")


## Build Initialized Model


In [ ]:
# Cell 8: Build and Save the Initialized Dynamic Model

InitializationHelpers.om_send(DynamicOMC, "deleteClass($INITIALIZED_MODEL)")
InitializationHelpers.om_send(DynamicOMC, "clearMessages()")
InitializationHelpers.om_send(DynamicOMC, "copyClass($MODEL, \"$INITIALIZED_MODEL\")")

InitializationHelpers.apply_initialization_modifiers!(
    DynamicOMC,
    INITIALIZED_MODEL,
    initializable_components,
    InitializationDictionaries.INIT_PARAMS,
    init_values_by_component,
    INIT_MODEL_BY_COMPONENT,
)

InitializationHelpers.om_send(DynamicOMC, "saveModel(\"$INITIALIZED_FILE_PATH\", $INITIALIZED_MODEL)")
println("Wrote initialized model: ", INITIALIZED_FILE_PATH)


## Optional Validation


In [ ]:
# Cell 9: Simulate the Initialized Dynamic Model

InitializedOMC = OMJulia.OMCSession()
InitializationHelpers.om_send(InitializedOMC, "loadModel(Complex)")
InitializationHelpers.om_send(InitializedOMC, "loadModel(ModelicaServices)")
ModelicaSystem(InitializedOMC, INITIALIZED_FILE_PATH, INITIALIZED_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
InitializationHelpers.om_send(InitializedOMC, "clearMessages()")
println("Checking the initialized model...")
chk_initialized = InitializationHelpers.om_send(InitializedOMC, "checkModel($INITIALIZED_MODEL)", parsed = false)
println(chk_initialized)

initialized_resultfile_prefix = INITIALIZED_MODEL

sim_result = sendExpression(
    InitializedOMC,
    "simulate($INITIALIZED_MODEL, outputFormat=\"csv\", fileNamePrefix=\"$initialized_resultfile_prefix\")",
    parsed = false,
)
println(sim_result)

initialized_resultfile = joinpath(
    getWorkDirectory(InitializedOMC),
    initialized_resultfile_prefix * "_res.csv",
)
println("Initialized result file: ", initialized_resultfile)


In [ ]:
# Cell 10: Plot the Initialized Dynamic Model

initialized_df = DataFrame(CSV.File(initialized_resultfile))

plotlyjs()
p = plot(initialized_df[!, "time"], initialized_df[!, PLOT_VARIABLE], label = [PLOT_VARIABLE])
plot!(p, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p, "Initialized dynamic model response")
xlabel!(p, "Time (s)")
ylabel!(p, PLOT_VARIABLE)
